# 04_00 Building the model (overview)

*📖 Book: chapter 4*

**How to use this notebook:** click a cell and press Shift+Enter to run just that cell.
Before each code cell, jot down what you think it will do. Then run it and see if you were right.

Run the cells in order, from the top. **A cell can use what earlier cells made**, so if you skip ahead you'll get `NameError: name '…' is not defined`. If that happens, run the cells above it, or just click Run All.

> **Where this fits**
>
> ```
> Odyssey text → numbers → [model reads] → model learns → model writes → MCP tool
> ```
>
> You are here: **model reads**
>
> Project files this connects to:
>
> - `src/homer_ai/model.py`
> - `src/homer_ai/gpt.py`
>
> **This notebook:** Attention alone is not a full model. A GPT stacks identical blocks, each with attention, a feed-forward step, layer norms and shortcuts; our `model.py` has none of these yet.
>
> For the big picture, what we're building and why, see the [course map](../00_start_here/00_00_course_map.ipynb).

## 1. What you'll learn

- Why attention alone isn't a full model
- The picture of a full GPT: a stack of identical blocks
- What our small `model.py` has, and what the full design adds
- What a config is, and why one config beats numbers scattered around the code
- Which notebook to open for each part

## 2. Words you'll meet

| Word | Meaning in plain English | Example |
|---|---|---|
| **GPT** | a popular design for language models: embeddings, then a stack of blocks, then a final layer that scores every token | GPT-2 (the book builds a small one) |
| **block** | one floor of the model: an attention step and a feed-forward step, each with a layer norm before it and a shortcut around it; a GPT stacks many identical blocks | our small model has none; GPT-2 small has 12 |
| **config** | short for configuration: one small table that holds every size of the model, so the numbers live in one place | `{"emb_dim": 32}` |

## 3. The idea

### ❓ Isn't attention already a whole model?

Almost, for a toy. Here's the picture for this chapter. Our `model.py` is a hut with one room, and a GPT is a tall building where every floor has the same layout. On each floor the same two things happen: the letters look back at each other (that's attention), and then each one thinks alone about what it heard. The numbers climb from floor to floor, so the first floors can pick up simple things, such as which letters often go together, and the higher floors build on those to pick up bigger patterns. One floor is called a **block**. The picture breaks in one place: in a real building an architect gives each floor its job, and here all the floors are built the same and only training makes them differ.

The hut, our `model.py`, is an embedding, **one** attention step, and a last layer that makes the 137 scores:

```
letters → embedding → attention → head → 137 scores
```

The building is drawn in [docs/neural-network-building.html](../../docs/neural-network-building.html), and in the next question. And a block is more than attention. It also has a few helper parts that keep the numbers well-behaved, so that a tall stack can still learn.

### ❓ What does a full GPT look like?

Read it from the bottom to the top, for the five letters of `ἄνδρα` ("man"). Each box is built in a notebook of this chapter, and the small grey names, like `04_01`, tell you which one. Five words appear in the picture:

- **layer norm**: keeps the numbers steady
- **attention**: looks back at the earlier letters
- **feed-forward**: lets each letter think on its own
- **output head**: makes one score for every token
- the `+`, a **shortcut**: the floor's input is added back to its output

Every block takes numbers in one shape and gives them back in the same shape, so blocks stack like identical floors.

**Look at:** the picture from the bottom (the letters) to the top (the scores). The orange floor is the one that repeats.

In [ ]:
import matplotlib.pyplot as plt

from homer_ai.studio_style import (
    ACCENT,
    INK,
    MUTED,
    arrow,
    box,
    chips,
    diagram_axes,
    plus,
    studio_figure,
    tag,
)

fig = studio_figure("A full GPT", "Floors, from letters up to scores", (9, 11.5))
ax = diagram_axes(fig, xlim=(-0.8, 9.2))
X = 4.2  # the middle of the building

# ground: the five letters of ἄνδρα as token IDs
chips(ax, X - 2.2, 0.85, ["58", "38", "29", "42", "26"], 0.8, 0.1)
for i, letter in enumerate("ἄνδρα"):
    tag(ax, X - 1.8 + i * 0.9, 0.3, letter, INK, 12, mono=True)
tag(ax, 7.0, 0.85, "token IDs", ha="left")

# floor 1: token embedding + position embedding
arrow(ax, (X, 1.4), (X, 1.85))
tag(ax, X + 0.2, 1.62, "(1, 5)", ha="left", mono=True)
box(ax, 2.6, 2.45, 2.7, 0.8, "token embedding", fontsize=10)
box(ax, 5.8, 2.45, 2.7, 0.8, "position embedding", fontsize=10)
plus(ax, 4.2, 2.45, 0.22)
tag(ax, 2.6, 1.95, "02_05")
tag(ax, 5.8, 1.95, "02_06")

# the block
arrow(ax, (X, 2.95), (X, 3.35))
tag(ax, X + 0.2, 3.15, "(1, 5, 32)", ha="left", mono=True)
box(ax, X, 5.15, 6.2, 3.65, color=ACCENT, lw=2.4, fill="none")
tag(ax, 1.4, 6.65, "BLOCK 1", ACCENT, 11, ha="left", weight="bold")
tag(ax, 7.1, 6.65, "04_04", ACCENT, 8, ha="right")
inner = [
    ("layer norm", "04_01"),
    ("attention", "03_04"),
    ("+", "04_03 shortcut"),
    ("layer norm", "04_01"),
    ("feed-forward", "04_02"),
    ("+", "04_03 shortcut"),
]
for i, (name, where) in enumerate(inner):
    y = 3.75 + i * 0.5
    if name == "+":
        plus(ax, 3.6, y, 0.2)
    else:
        box(ax, 3.6, y, 3.0, 0.38, name, fontsize=9.5)
    tag(ax, 5.4, y, where, MUTED, 8, ha="left")
    if i > 0:
        arrow(ax, (3.6, y - 0.3), (3.6, y - 0.21), head=8)

# more floors, the same again
arrow(ax, (X, 7.05), (X, 7.55))
tag(ax, X + 0.2, 7.3, "(1, 5, 32)", ha="left", mono=True)
text = "BLOCK 2 … BLOCK N: the same floor again"
box(ax, X, 7.95, 6.2, 0.7, text, MUTED, fontsize=9.5)

# the roof
arrow(ax, (X, 8.3), (X, 8.65))
box(ax, X, 8.95, 3.0, 0.5, "final layer norm", fontsize=10)
tag(ax, 6.1, 8.95, "04_01", ha="left")
arrow(ax, (X, 9.2), (X, 9.5))
box(ax, X, 9.8, 3.0, 0.5, "output head", fontsize=10)
tag(ax, 6.1, 9.8, "04_05", ha="left")
arrow(ax, (X, 10.05), (X, 10.4))
tag(ax, X + 0.2, 10.22, "(1, 5, 137)", ha="left", mono=True)
box(ax, X, 10.75, 4.6, 0.6, "137 scores for each letter", fontsize=10)
plt.show()

### ❓ What does our `model.py` have, and what's missing?

Let's count the real parts and weights of our model first.

👉 **Predict first:** how many of the parts in the picture above does `HomerGreekLLM` have?

In [ ]:
from homer_ai.model import HomerGreekLLM

model = HomerGreekLLM(137, embed_dim=32)
for name, layer in model.named_children():
    count = sum(p.numel() for p in layer.parameters())
    print(f"{name:10} {count:>6,} weights")
print(f"{'total':10} {sum(p.numel() for p in model.parameters()):>6,} weights")

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

Five layers: the embedding, query, key, value and `head`. That's 11,840 weights in all. The `forward` method also builds the causal mask, which has no weights.

Three words for the numbers we just counted. A **weight** is one learned number in a layer's table. A bias is the extra learned number a layer can add to each output ([01_02](../01_foundations/01_02_neural_networks.ipynb)). And a parameter is PyTorch's word for either. `.parameters()` lists all of them, so summing it counts everything the model can learn. Our layers use `bias=False`, so here weights and parameters are the same 11,840 numbers.

So how far is that from a full GPT? Here's the whole comparison:

| Part | Our small `model.py` | The full design |
|---|---|---|
| Token embedding | yes (137 × 32) | yes |
| Position embedding | **no** ([02_06](../02_text_to_numbers/02_06_positions.ipynb)) | yes (24 × 32) |
| Attention | one head, mask | several heads ([03_04](../03_attention/03_04_many_heads.ipynb)), mask, and an output layer |
| Feed-forward | **no** | yes, in every block |
| Layer norm | **no** | before attention, before feed-forward, and once at the end |
| Shortcuts | **no** | two in every block |
| Dropout | **no** | optional |
| Blocks | none: just one attention step | N identical blocks |
| Output head | yes (32 × 137) | yes |
| Weights | 11,840 | depends on the config (next question) |

</details>

### ❓ What's a config, and why do we need one?

A full model has many sizes: how many tokens, how many numbers per token, how many heads, how many blocks. In the book they all live in one small table, a Python dictionary called a **config**. Here's ours, for our 137 tokens.

👉 **Predict first:** one line sets how long a chunk of text the model reads at once. Which key is it?

In [ ]:
GPT_CONFIG = {
    "vocab_size": 137,      # how many different tokens: our 137 characters
    "context_length": 24,   # how many tokens the model reads at once
    "emb_dim": 32,          # how many numbers describe each token
    "n_heads": 2,           # how many attention heads per block
    "n_layers": 2,          # how many blocks are stacked
    "drop_rate": 0.1,       # dropout: how often to switch off a number while training
}
for key, value in GPT_CONFIG.items():
    print(f"{key:15} = {value}")

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

Six lines, six sizes. Let's take them one at a time:

- `vocab_size`: the number of tokens, 137 for our Book 1 ([02_01](../02_text_to_numbers/02_01_tokens_and_vocabulary.ipynb)). It sets the size of the first table and of the last layer.
- `context_length`: how many letters the model reads at once, 24 ([02_04](../02_text_to_numbers/02_04_sliding_window.ipynb)). It sets the number of rows in the position table.
- `emb_dim`: the numbers per letter, 32 ([02_05](../02_text_to_numbers/02_05_token_embeddings.ipynb)). Almost every layer in the model is this wide.
- `n_heads`: attention heads per block ([03_04](../03_attention/03_04_many_heads.ipynb)). It must divide `emb_dim`: 32 numbers split between 2 heads of 16.
- `n_layers`: how many blocks sit on top of each other. Our `model.py` has none, GPT-2 small has 12.
- `drop_rate`: dropout, from [03_03](../03_attention/03_03_hiding_the_future.ipynb). 0.1 means 1 number in 10 is switched off while training.

</details>

### ❓ Why not just write the numbers where we use them?

Because then the same number sits in several files, and when you change it you always forget one. We can find the repeats in our real project right away.

👉 **Predict first:** do you think the `32` and the `24` each appear once, or in several places?

In [ ]:
from pathlib import Path

import homer_ai
from homer_ai import train_homer

# we only READ these files as text
folder = Path(homer_ai.__file__).parent
for path in (Path(train_homer.__file__), folder / "model.py", folder / "lessons.py"):
    for number, line in enumerate(path.read_text().splitlines(), start=1):
        if "embed_dim=32" in line or "chunk_size=24" in line or "[-24:]" in line:
            print(f"{path.name}:{number}: {line.strip()}")

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

So `embed_dim=32` is written in four places (`train_homer.py`, `model.py` as a default, and twice in `lessons.py`), and the window of 24 sits on its own as `chunk_size=24`. Nothing links them. Change one and forget the others, and the model loads with the wrong shapes, or quietly reads the wrong window. (The MCP server doesn't repeat them: a checkpoint carries its own config, [05_06](../05_training_on_the_odyssey/05_06_saving_and_loading.ipynb), and the server builds the model from that.) With a config, every file reads `GPT_CONFIG["emb_dim"]`. Change it once, and everything follows. (The GPT does it this way: `GPTConfig` in `src/homer_ai/gpt.py` is the one place. The notebooks here keep their own copy of a dictionary so that each one runs alone.)

*In .NET we'd call this the options pattern: settings in one object, passed around, never copied.*

</details>

### ❓ Which notebook do I open for each part?

| Notebook | Question it answers |
|---|---|
| [`04_01_layer_norm.ipynb`](04_01_layer_norm.ipynb) | How do we stop the numbers drifting too big or too small? |
| [`04_02_feed_forward.ipynb`](04_02_feed_forward.ipynb) | What does each letter do with what attention gave it? |
| [`04_03_shortcuts.ipynb`](04_03_shortcuts.ipynb) | How do learning signals reach the lower floors of a tall stack? |
| [`04_04_one_block.ipynb`](04_04_one_block.ipynb) | How do the parts fit together into one block? |
| [`04_05_the_full_model.ipynb`](04_05_the_full_model.ipynb) | How do we stack blocks into the whole model, and make it write? |

### ❓ Will we train this bigger model in this chapter?

No. Chapter 4 only builds it, with random numbers, and checks that the shapes fit. Training it is [chapter 5](../05_training_on_the_odyssey/05_00_chapter_overview.ipynb)'s job. And we don't touch `model.py` here. The full model gets a file of its own, `src/homer_ai/gpt.py` ([04_05](04_05_the_full_model.ipynb)).

## 4. Recap

- A full GPT is a stack of identical blocks between the embeddings and the output head. A block is attention plus feed-forward, each with a layer norm and a shortcut.
- Our `model.py` has the embedding, one attention head and the head (11,840 weights). It has no positions, norms, feed-forward or shortcuts.
- A config puts every size in one place. Without one, `32` is repeated across three files.

**Next:** [04_01: Layer norm](04_01_layer_norm.ipynb): keeping the numbers from drifting.